# Building Makemore
Now we are going to be building a character level model that makes use of "bigrams" which are basically just pairs of words. We will train this character level language model to makemore of human names taken from the SSA, specifically 

"...The included names.txt dataset, as an example, has the most common 32K names takes from ssa.gov for the year 2018." (From https://github.com/karpathy/makemore)

We begin by showing an example of the bigrams in the first name after adding start and end tokens.

In [ ]:
with open("names.txt") as f:
    print(f.readline()) # show a single name from the list of 32k names
    all_names = [name.rstrip('\n') for name in f.readlines()] 

# print(all_names)

name1 = ['<s>'] + list(all_names[0]) + ['<e>']
print(name1)

for c1, c2 in zip(name1, name1[1:]):
    print(c1,c2)

# Making a Bigram Dictionary
We're now going to use these names to examine statistical properties of the bigrams that exist in this dataset, namely the number of occurences.

In [ ]:
bigram_dict = {}

for name in all_names:
    name_with_special_tokens = ['<s>'] + list(name) + ['<e>']
    for c1, c2 in zip(name_with_special_tokens, name_with_special_tokens[1:]):
        bigram = (c1,c2)
        bigram_dict[bigram] = bigram_dict.get(bigram, 0) + 1 # increment the count of this specific bigram

print(bigram_dict)

In [ ]:
# Which bigrams were the most frequent?

# items() turns this into a list of [ ((c1,c2), counts), ... ]
# then we used the sorted key param to organize this by the counts instead of the tuple
sorted(bigram_dict.items(), key=lambda kv: -kv[1])

Now we see the most common bigrams are `n<e>`, `a<e>`, and `an`. It is very common for a name
to end with `n` or `a`. Because we are going to be using `pytorch` it will be much easier if we convert this into a tensor instead of a python dictionary. We do that below.

In [ ]:
all_names_as_str = ''.join(all_names) # combine all names into one string
all_letters = set(all_names_as_str) # remove dupes
list_of_letters = sorted(list(all_letters))

# Now make a mapping so we can represent our data as a 2x2 tensor with the counts as the cell entries
stoi = {letter: interger_representation for interger_representation, letter in enumerate(list_of_letters)}
stoi['<s>'] = 26
stoi['<e>'] = 27
stoi

In [ ]:
import torch
import matplotlib

In [ ]:
N = torch.zeros(len(stoi),len(stoi), dtype=torch.int32)

for name in all_names:
    name_with_special_tokens = ['<s>'] + list(name) + ['<e>']
    for c1, c2 in zip(name_with_special_tokens, name_with_special_tokens[1:]):
        N[stoi[c1], stoi[c2]] += 1 # Increment the cell for this bigram lookup in the tensor

print(N[stoi['a'], stoi['n']])
print(bigram_dict[('a', 'n')])
        
# They match!